# Predicting buying pressure on a security, and testing it for alpha

A **security-level** study, separate from the fund-level replication.

| | |
|---|---|
| unit of observation | (security, quarter) |
| **target** | `buy_frac(s, q)` = # funds buying s / # funds owning s, over the window q → q+1 |
| **features** | latest Barra `GEMLT_*` exposures at quarter end, past security returns, **lagged** buying pressure and ownership breadth |
| **alpha test** | rank securities by *predicted* buying pressure, then look at forward returns |

## The timing trap, stated once

Write $\mathcal I_t$ for everything knowable once the quarter-$t$ holdings snapshot exists.
`buy_frac` over q → q+1 compares holdings at q against holdings at q+1, so

$$\text{buy\_frac}(s,q)\ \text{is}\ \mathcal I_{q+1}\text{-measurable, NOT}\ \mathcal I_q\text{-measurable}$$

even though the column sits physically on the q row — exactly like `future_1q_ret` does.
**A column living on row q says nothing about when its value became knowable.**

Two consequences, both enforced in code:

- **only LAGGED `buy_frac` may be a feature.** `buy_frac_lag1` = buy_frac(q−1) closed at q, so
  it is $\mathcal I_q$-measurable. The contemporaneous `buy_frac(q)` is *not*, and is excluded.
- **the target is buy_frac(q) itself** — the buying over q → q+1. It is strictly after every
  feature, so this is not look-ahead, and it is the *tightest honest* choice. Shifting the
  target one further quarter would skip a whole quarter for no gain in validity and would cut
  the attainable IC from $\rho_1$ to $\rho_2$ (≈ $\rho_1^2$ for a persistent series).

### Which return horizon is honest here

Horizons are named by the **decision point**, not the return window, because what makes a
horizon valid is *when the sort variable became knowable*.

`pred_buy_frac` is $\mathcal I_q$-measurable — it exists the moment quarter q closes. Trading on
it at that close and holding through q+1 earns `fwd_1q`, and **the entire return window lies
after the forecast**. No overlap, no bias.

| | return window | decision point | status |
|---|---|---|---|
| `quarter_end` | q → q+1 | close of q | **unbiased, and the headline** ← default |
| `one_q_delay` | q+1 → q+2 | close of q, wait a quarter | only if holdings arrive late |
| `two_q_delay` | q+2 → q+3 | wait two quarters | full ~45–60 day filing delay + a quarter |

The legacy names `contemporaneous` / `predictive` / `tradeable` still work and map onto these
three in order.

> **The one asymmetry.** `alpha_actual` ranks on *realised* buying, which is
> $\mathcal I_{q+1}$-measurable. At `quarter_end` that **does** overlap `fwd_1q` and is biased —
> read it as a perfect-foresight *ceiling*, never as a strategy. `alpha_pred` has no such
> problem at any horizon.

### Does `quarter_end` assume too much?

It assumes $S_q$ is in hand at the q close, since `buy_frac_lag1` needs it. True for internal
holdings data; **not** true if the data goes through a filing delay — in that case the newest
usable lag at the q close is buy_frac(q−2), and every feature lag must shift out by one.

## Joining Barra to the holdings

The Barra file carries **`stkid`**, which is the same identifier the holdings panel calls
**`security`** — so the join is direct, no mapping table. Both sides pass through `_norm_id`
first, because one may be an int and the other a zero-padded string (`'0010001'`, `'10001.0'`
and `10001` all denote the same security).

Coverage is reported **three ways**, because a healthy-looking row rate can hide a
systematic gap:

| | what it catches |
|---|---|
| distinct securities in both | a class of securities missing from Barra entirely |
| security-quarters hit | the overall row-level rate |
| **per-quarter hit rate (min / median / max)** | whole quarters missing — e.g. Barra starting later than the holdings |

Below `min_match_rate` the loader raises rather than producing a near-empty panel.

## Models

`gbm` (history via explicit lags), `lstm` (one sample = a security's last 8 quarters `[T, F]`),
`ridge` (linear benchmark). Same rolling split for all three.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from dataclasses import replace
%load_ext autoreload
%autoreload 2
import buy_pressure as B
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
B.check_version()

## 0. Inspect the Barra pickle first

Before anything else, look at the actual structure — column names, how the `GEMLT_` factors
are laid out, and whether `day` / `sedol` are columns or part of the index.

In [ ]:
raw = B.inspect_barra(B.Config())

## 1. Configuration

Nothing to set for the join — Barra's `stkid` is the holdings `security` id, and both sides
are normalised so an int and a zero-padded string still match. `barra_id_col` only needs
changing if that column is named differently in your copy of the file.

Two knobs that do matter:

- **`max_rank = None`** keeps the whole book. Buying pressure is a *breadth* measure
  (how many owners are buying), so truncating to the top-N positions would bias it toward
  large holdings.
- **`min_owners = 5`** drops securities with too few owners for `buy_frac` to mean anything
  — with 2 owners it can only be 0, 0.5 or 1.

In [ ]:
CFG_KW = dict(
    holdings_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    barra_path    = "manager_holdings/barra_GEMLTL_R3000_Prod_new_MSCI_wkly.pickle",

    # --- the join: Barra's `stkid` IS the holdings `security` id ---
    barra_id_col   = "stkid",
    min_match_rate = 0.05,          # raise if fewer than this share of rows carry exposures

    inv_type_codes = (401,),
    max_rank    = None,             # None = whole book; buying pressure is a breadth
                                    # measure, so truncating the book distorts it
    min_owners  = 5,                # a buy_frac from 2 funds is noise
    min_quarters = 12,
    eval_timing = "predictive",
    window_q = 28, test_q = 8, step = 8,
    model = "gbm",
    seq_len = 8, hidden = 64, max_epochs = 40, batch = 4096, device = "auto",
)
known = set(B.Config.__dataclass_fields__)
dropped = {k: v for k, v in CFG_KW.items() if k not in known}
BASE = B.Config(**{k: v for k, v in CFG_KW.items() if k in known})
print("!! unsupported by this copy:", list(dropped)) if dropped else print("all keys accepted")
BASE

## 2. Build the panel

Four printed diagnostics decide whether anything downstream is meaningful:

- **`[hold] buy_frac mean / sd`** — with a tiny sd there is no cross-sectional variation to predict
- **`[hold] median owners`** — few owners per security makes the target mostly noise; raise `min_owners`
- **`[barra] N weekly rows -> M security-quarters`** — should shrink by roughly 13x (weeks per quarter)
- **`--- id join coverage ---`** — read the **per-quarter min**, not just the overall rate.
  A quarter at 0% means Barra has no data there and those rows carry NaN exposures; the tree
  models tolerate that, but the affected sample is effectively running on the
  buying-pressure and return features alone.

In [ ]:
panel = B.build_panel(BASE)
feats = B.feature_list(panel, BASE)
print(f"\n{len(feats)} features")
print("  Barra exposures :", [f for f in feats if f.startswith(BASE.barra_prefix)][:12])
print("  buying history  :", [f for f in feats if "buy_frac" in f or "sell_frac" in f or "owning" in f])
print("  returns / other :", [f for f in feats if f.startswith("ret_") or f.startswith("log_") or f == "w_mean"])

# Where the Barra exposures are actually present, by quarter. If early quarters are empty,
# consider dropping them rather than training on rows whose factor block is all NaN.
gem = [f for f in feats if f.startswith(BASE.barra_prefix)]
cov = panel.groupby("yq")[gem].apply(lambda d: d.notna().any(axis=1).mean())
print(f"\nBarra coverage by quarter: min={cov.min():.1%} median={cov.median():.1%} max={cov.max():.1%}")
display(cov.to_frame("barra_coverage").T.round(2))

In [ ]:
display(panel[["buy_frac", "sell_frac", "n_owning", "target_buy_frac"]]
        .describe().loc[["count","mean","std","min","25%","50%","75%","max"]].round(4))
print("target availability by quarter (tail):")
display(panel.groupby("yq")["target_buy_frac"].agg(["size", "mean"]).tail(8).round(4))

---
## 3. GBM

In [ ]:
RESULTS = {}
RESULTS["gbm"] = B.run_one(panel, replace(BASE, model="gbm"), "gbm")
B.free(RESULTS)

In [ ]:
r = RESULTS["gbm"]
print("### prediction quality -- is buying over q -> q+1 predictable from what is known at q?")
print("    the row that matters is 'model minus naive': naive = buy_frac_lag1, same information set")
display(r["quality"].round(4))
print("### alpha: securities ranked by PREDICTED buying pressure (all horizons unbiased)")
for tm in ("quarter_end", "one_q_delay", "two_q_delay"):
    print(f"-- {tm} --"); display(r["alpha_pred"][tm].round(3))
print("### CEILING: ranked by REALISED buying = the target itself (perfect foresight, NOT a strategy)")
print("    at quarter_end this overlaps the return window and is biased -- read it as an upper bound")
for tm in ("quarter_end", "one_q_delay", "two_q_delay"):
    print(f"-- {tm} --"); display(r["alpha_actual"][tm].round(3))


---
## 4. LSTM

One sample is a security's last `seq_len` quarters of features. Sequences are assembled from
indices rather than materialised, so memory stays near the size of the feature matrix. Set
`lstm_max_train` if a CPU-only run drags.

In [ ]:
RESULTS["lstm"] = B.run_one(panel, replace(BASE, model="lstm"), "lstm")
B.free(RESULTS)

In [ ]:
r = RESULTS["lstm"]
display(r["quality"].round(4))
for tm in ("quarter_end", "one_q_delay", "two_q_delay"):
    print(f"-- alpha, predicted buying / {tm} --"); display(r["alpha_pred"][tm].round(3))


---
## 5. Ridge — linear benchmark

If ridge and gbm land in the same place, the signal is broadly linear and the tree is not
inventing structure.

In [ ]:
RESULTS["ridge"] = B.run_one(panel, replace(BASE, model="ridge"), "ridge")
B.free(RESULTS)

In [ ]:
display(RESULTS["ridge"]["quality"].round(4))
display(RESULTS["ridge"]["alpha_pred"][BASE.eval_timing].round(3))

---
## 6. Comparison

Two separate questions, and they can have different answers:

1. **Is buying pressure predictable at all?** → `rank_IC` versus `naive_IC`. The naive
   benchmark is `buy_frac_lag1`, i.e. *"assume this quarter looks like last quarter"*. It sits
   on **exactly** the information set the model is given, so beating it is the real test.
   The quality table reports the difference directly — that difference, not `rank_IC`, is the
   number to quote.
2. **Does predicted buying carry alpha?** → the `alpha_*` columns. `quarter_end` is the
   headline: the forecast exists at the q close and the return window opens after it.

A high `rank_IC` with a near-zero edge over naive means the model has learned only that
buying pressure repeats — worth saying plainly rather than reporting the raw IC.


In [ ]:
cmp = B.compare(RESULTS)
cmp

## Save

In [ ]:
import os
os.makedirs("outputs_buy_pressure", exist_ok=True)
for tag, r in RESULTS.items():
    if not isinstance(r, dict) or "quality" not in r: continue
    r["quality"].to_csv(f"outputs_buy_pressure/quality_{tag}.csv", index=False)
    for tm in ("quarter_end", "one_q_delay", "two_q_delay"):
        r["alpha_pred"][tm].to_csv(f"outputs_buy_pressure/alpha_pred_{tag}_{tm}.csv", index=False)
        r["alpha_actual"][tm].to_csv(f"outputs_buy_pressure/alpha_actual_{tag}_{tm}.csv", index=False)
cmp.to_csv("outputs_buy_pressure/compare_models.csv", index=False)
print("saved to outputs_buy_pressure/")


## Reading the result honestly

- **`model minus naive` near zero** → the features add nothing beyond persistence. A high
  `rank_IC` on its own does not rescue this; buying pressure is autocorrelated, so a large IC
  is the baseline, not the achievement.
- **IC beats naive but no alpha** → buying is forecastable yet carries no return information.
  A clean, reportable negative result.
- **Negative `Q5-Q1`** → securities predicted to be bought subsequently *underperform*. Before
  calling this alpha, rule out the rebalancing mechanic: a weight-targeting fund **adds**
  shares when a stock falls, so "buy" is mechanically tied to a low same-window return. Run
  `diagnose_buy_alpha.py` — if the sign lives only at `quarter_end` and dies at `one_q_delay`,
  it is that mechanic, not a forecast.
- **Check the ceiling first.** If `alpha_actual` at `two_q_delay` is flat, then even perfect
  foresight of buying earns nothing there, and no forecast can do better. That bounds the whole
  exercise and is itself the finding.
- **Check `monotone`.** A large spread with a non-monotone Q1→Q5 pattern usually means one
  extreme quintile is doing all the work, and the t-stat is not trustworthy.

Returns here are gross — no trading costs, and a buying-pressure sort implies turnover, so any
spread should be discounted before it is called alpha.
